# 🧊 Reasoning Segmentation for Sea-Ice SAR — Colab Training

**What this notebook trains** — the redesigned *reasoning-segmentation* pipeline, where **text and image jointly navigate segmentation**:

1. **Per-image annotator descriptions** (`dataset/*/descriptions/`) are now the model's text input — not a constant prompt.
2. The U-Net mask decoder is **text-guided**: the description's fused tokens are injected at the bottleneck, the sentence embedding FiLM-modulates the bottleneck features, and a **text–pixel similarity map** is fed to the mask head. The text decides *where* to segment.
3. **Ground truth changed**: the raw `_scat` scattering maps **are** the ground truth (per-image min–max normalised, continuous in [0,1]). Otsu-binarised masks are no longer used as labels. Binary metrics binarise the soft GT at 0.5.
4. Class names are **scrubbed** from descriptions by default (`scrub_class_names=True`) so the 6-class F1 stays leakage-free.

**Numbers to beat** (previous published run, measured against the old Otsu GT):

| Metric | Previous | Floor (all-foreground, new scat GT) |
|---|---|---|
| Segmentation mIoU | 0.351 | 0.330 |
| Classification weighted F1 | 0.778 | 0.167 (chance) |

> ⚠️ Runtime → **Change runtime type → GPU** (A100/L4 recommended; T4 works with a smaller batch).

## 1. 🖥️ Check GPU

In [ ]:
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout if r.returncode == 0 else 'No GPU found — switch the runtime type to GPU!')

## 2. 📂 Mount Google Drive (optional — persistent checkpoints)

In [ ]:
USE_DRIVE = True  # set False to keep everything in the ephemeral VM
DRIVE_DIR = '/content/drive/MyDrive/reasoning_seg'
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    import os; os.makedirs(DRIVE_DIR, exist_ok=True)
    print('Checkpoints will be copied to', DRIVE_DIR)

## 3. 📥 Clone the repository

In [ ]:
import os
REPO_URL = 'https://github.com/prakhar443/Reasoning_Segmentation_new.git'
BRANCH   = 'claude/charming-cerf-2rh1i7'  # reasoning-segmentation branch

if not os.path.exists('/content/Reasoning_Segmentation_new'):
    !git clone --branch {BRANCH} {REPO_URL} /content/Reasoning_Segmentation_new
%cd /content/Reasoning_Segmentation_new
!git log --oneline -3

## 4. 📦 Install dependencies (~3 min on first run)

In [ ]:
!pip install -q -r requirements.txt
import torch, albumentations, transformers
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
print('albumentations', albumentations.__version__, '| transformers', transformers.__version__)

## 5. 🔎 Sanity check — new GT + text channel

Verify the three pillars of the redesign before burning GPU hours:
the soft scat ground truth, the per-image descriptions, and the text-guided decoder flag.

In [ ]:
from config import cfg
print('mask_target_mode      :', cfg.data.mask_target_mode, '   (soft_scat = raw scat maps are GT)')
print('use_image_descriptions:', cfg.data.use_image_descriptions)
print('scrub_class_names     :', cfg.data.scrub_class_names)
print('text_guided_decoder   :', cfg.model.text_guided_decoder)

from data.dataset import SeaIceDataset
ds = SeaIceDataset('dataset', split='test', use_augmentation=False)
s = ds[0]
m = s['mask']
print('\ntest size:', len(ds))
print('mask: shape', tuple(m.shape), '| min %.3f max %.3f | continuous values: %d' % (m.min(), m.max(), len(m.unique())))
print('\nDescription that navigates this segmentation:')
print(' ', s['long_desc'][:400])

## 6. 📉 Degenerate floor under the new ground truth

All-foreground baseline against the **scat** GT — the number the trained model must clear on mIoU.

In [ ]:
!python baseline_allforeground.py --split test --gt scat

## 7. 🚀 Train

Defaults in `config.py` are the reasoning-segmentation configuration. Early stopping on val mIoU; best-mIoU and best-F1 checkpoints are saved separately.

*A100/L4*: leave `BATCH_SIZE=4`. *T4 (16 GB)*: use `BATCH_SIZE=2`.

In [ ]:
import torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else ''
BATCH_SIZE = 2 if 'T4' in gpu else 4
EPOCHS = 60
print(f'GPU: {gpu} → batch_size={BATCH_SIZE}, epochs={EPOCHS}')

!python train.py --output_dir outputs --batch_size {BATCH_SIZE} --epochs {EPOCHS}

## 8. 💾 Back up checkpoints to Drive

In [ ]:
if USE_DRIVE:
    !cp -v outputs/best_model.pth outputs/best_model_f1.pth {DRIVE_DIR}/ 2>/dev/null || true

## 9. 📊 Evaluate on the held-out test split

Metrics are computed against the **scat ground truth** (soft GT binarised at 0.5 of its normalised range).

In [ ]:
!python evaluate.py --checkpoint outputs/best_model.pth --output eval_results/ --save_visualizations --save_masks

## 10. 🏁 Compare against the previous model

In [ ]:
import json
rep = json.load(open('eval_results/test_evaluation_report.json'))

PREV = {'mean_iou': 0.351, 'ciou': 0.456, 'mean_dice': 0.442,
        'accuracy': 0.833, 'weighted_f1': 0.778}
FLOOR_MIOU = 0.3302  # all-foreground vs scat GT (section 6)

rows = [
    ('Segmentation mIoU',        'mean_iou'),
    ('Segmentation cIoU',        'ciou'),
    ('Segmentation Dice',        'mean_dice'),
    ('Classification accuracy',  'accuracy'),
    ('Classification weighted F1','weighted_f1'),
]
print(f"{'Metric':<28}{'Previous':>10}{'This run':>10}{'Δ':>9}")
print('-' * 57)
for name, key in rows:
    new, old = rep.get(key, float('nan')), PREV.get(key, float('nan'))
    print(f'{name:<28}{old:>10.3f}{new:>10.3f}{new - old:>+9.3f}')
print('-' * 57)
print(f"All-foreground floor (scat GT) mIoU: {FLOOR_MIOU:.3f} → "
      f"{'CLEARED ✅' if rep['mean_iou'] > FLOOR_MIOU else 'not cleared ❌'}")
print('\nNote: previous mIoU was measured against the old Otsu-binary GT;')
print('this run is measured against the raw-scat GT (the new label definition).')

## 11. 🖼️ Qualitative results — text-navigated masks

Image · soft scat GT · prediction, with the description that guided each mask.

In [ ]:
import torch, textwrap
import matplotlib.pyplot as plt
from config import cfg
from data.dataset import SeaIceDataset, collate_fn
from models.pipeline import SeaIceSegmentationPipeline

device = 'cuda' if torch.cuda.is_available() else 'cpu'
ckpt = torch.load('outputs/best_model.pth', map_location=device)
model = SeaIceSegmentationPipeline(cfg.model, use_sam=False).to(device)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

ds = SeaIceDataset('dataset', split='test', use_augmentation=False)
idxs = [0, 15, 30, 45, 60, 75]   # two per a few classes
fig, axes = plt.subplots(len(idxs), 3, figsize=(12, 3.2 * len(idxs)))
for r, i in enumerate(idxs):
    b = collate_fn([ds[i]])
    with torch.no_grad():
        out = model(images=b['image'].to(device), descriptions=b['long_desc'])
    img = b['image'][0].permute(1, 2, 0).cpu().numpy()
    gt  = b['mask'][0, 0].cpu().numpy()
    pr  = out['masks'][0, 0].cpu().numpy()
    axes[r, 0].imshow(img); axes[r, 0].set_title('SAR image', fontsize=9)
    axes[r, 1].imshow(gt, cmap='viridis', vmin=0, vmax=1); axes[r, 1].set_title('GT: raw scat map', fontsize=9)
    axes[r, 2].imshow(pr, cmap='viridis', vmin=0, vmax=1); axes[r, 2].set_title('Prediction', fontsize=9)
    for ax in axes[r]: ax.axis('off')
    axes[r, 0].text(0, -18, textwrap.fill('“' + b['long_desc'][0][:140] + '…”', 110), fontsize=7)
plt.tight_layout(); plt.savefig('eval_results/qualitative.png', dpi=120); plt.show()

## 12. 🧭 Does the text actually navigate the mask?

Same image, different queries → the mask should change. This is the reasoning-segmentation behaviour the old constant-prompt model could not exhibit.

In [ ]:
queries = [
    ds.samples[0]['long_desc'],                                   # true description
    'Segment the dark open-water areas with weak backscatter.',   # opposite query
    'Segment the bright, heavily ridged ice with strong backscatter.',
]
b = collate_fn([ds[0]])
fig, axes = plt.subplots(1, len(queries) + 1, figsize=(4 * (len(queries) + 1), 4))
axes[0].imshow(b['image'][0].permute(1, 2, 0).cpu().numpy()); axes[0].set_title('SAR image', fontsize=9); axes[0].axis('off')
for c, q in enumerate(queries):
    with torch.no_grad():
        out = model(images=b['image'].to(device), descriptions=[q])
    axes[c + 1].imshow(out['masks'][0, 0].cpu().numpy(), cmap='viridis', vmin=0, vmax=1)
    axes[c + 1].set_title(textwrap.fill(q[:80], 32), fontsize=7); axes[c + 1].axis('off')
plt.tight_layout(); plt.show()

## 13. 📝 Notes — squeezing out more performance

- **More epochs**: bump `EPOCHS` in section 7; early stopping (patience 12 on val mIoU) prevents wasted compute.
- **Unredacted text**: `cfg.data.scrub_class_names = False` lets full descriptions through — segmentation usually gains a little and classification F1 rises sharply, but the F1 is then no longer leakage-free. Keep the default for honest numbers.
- **Wider decoder**: `cfg.model.decoder_base_channels = 48` if you are on an A100.
- **Legacy comparison**: set `cfg.data.mask_target_mode = 'binary'` to reproduce the old Otsu-GT training.
- The best-F1 checkpoint is saved separately as `outputs/best_model_f1.pth`.